# M40 — Evaluate AI Systems Systematically

M34 already grounds answers. M39 already routes, remembers, and fails closed.
The useful whole here is an **evaluation harness**, not another model and not
an M41 architecture diagram:

frozen eval pack → invoke M34/M39/M37 traces → deterministic graders →
slices and proxies → release gate.

Write the cases before optimizing either system. A high average is not
permission to ship a rare unsupported citation, a schema-invalid tool call,
a mislabeled complete, or a double ledger post.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a case id, a `fail_reasons` tuple, a slice rate,
an `eval_version`, or a localized grader string.

Do not retune M34 holdout ids to raise the average. Do not drop the only
failing case and relabel the rest as holdout. The repository does not prefill
learner answers, ADR text, or competence.

Canonical sources (named, not imported): `anthropic-evals`, `anthropic-agents`.
Content bundle: `ai-system-evals`.

P7 lists this mission as phase-end. **V11 does not close** because the package
exists. Architecture remains M41. Implementation is not learner completion.


In [ ]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M40" / "evaluation_harness.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M34.rag_pipeline import (
    EVAL_VERSION as M34_EVAL_VERSION,
    answer_labeled,
    evaluate_set,
    verify_support,
)
from missions.M39.robust_agent import (
    TRACE_FIELDS as M39_TRACE_FIELDS,
    run_robust_task,
)
from missions.M40.evaluation_harness import (
    AGGREGATE_ONLY_POLICY,
    CANONICAL_POLICY,
    DETERMINISTIC_GRADERS,
    EVAL_VERSION,
    HARNESS_VERSION,
    OBSERVABILITY_FIELDS,
    SCALE_LIMIT,
    SCENARIO_TAXONOMY,
    SEED,
    SYSTEM_MAP,
    EvalVersionError,
    OptionalLLMJudgeUnavailable,
    ablate_trace,
    aggregate_report,
    calibrate_rubric,
    decide_release_gate,
    grade_citation_support,
    grade_idempotency,
    grade_state_termination,
    grade_tool_schema,
    graph_public,
    handoff_contract,
    inject_regression,
    invoke_case,
    load_eval_pack,
    load_expected_payload,
    load_rubric_labels,
    optional_llm_judge,
    outcome_success,
    pipeline_with_defect,
    repair_run,
    run_suite,
    slice_rates,
)

print("repository root:", ROOT)
print("harness version:", HARNESS_VERSION)
print("eval version:", EVAL_VERSION)
print("seed:", SEED)
print("scale:\n", SCALE_LIMIT)


## What M34 and M39 already handed you

M34: `answer_labeled` / `evaluate_set` / `verify_support`, with citations,
abstention, and failure layers. Do not retune that synthesizer here.

M39: `run_robust_task` with retrieved ids, route, attempts, degraded,
circuit, and posted amount. Do not reopen memory policy here.

M34 → M40 consumes grounded-answer traces as `missions.M34.rag_pipeline`.
M39 → M40 consumes robust-agent traces as `missions.M39.robust_agent`.
Tool schema and idempotency graders also see M37's registry. M41 is still closed.


In [ ]:
pack = load_eval_pack(require_canonical=True)
print("scenario taxonomy:")
print(json.dumps({key: list(value) for key, value in SCENARIO_TAXONOMY.items()}, indent=2))
print("deterministic graders:", DETERMINISTIC_GRADERS)
print("observability fields:", OBSERVABILITY_FIELDS)
print("system map:\n", SYSTEM_MAP)
print("eval_version:", pack.eval_version)
print("n cases:", len(pack.cases))
print("families:", sorted({case.family for case in pack.cases}))
print("downloaded:", pack.downloaded)
print("network_required:", pack.network_required)
print("contaminated:", pack.contaminated)
print("held_out_untuned:", pack.held_out_untuned)
print("m34 eval version constant:", M34_EVAL_VERSION)
print("m39 trace fields:", M39_TRACE_FIELDS)
print("canonical policy name:", CANONICAL_POLICY.name)
print("aggregate policy name:", AGGREGATE_ONLY_POLICY.name)

try:
    optional_llm_judge("not-required")
except OptionalLLMJudgeUnavailable as exc:
    print("llm judge adapter:", type(exc).__name__)


The pack is a frozen **input**. Families, grader names, and version
are maps of the system, not scores. Outcome success, slice rates, and
the ship/no-ship bit are not in this cell.

Named sources (`anthropic-evals`, `anthropic-agents`) argue for
code-based graders on invariants, transcripts you can read, and
capability versus regression suites. They are not SDKs to import.


## Useful whole — run the frozen pack

**Predict before running.** Timestamp your prediction.

On current M34 and M39 fixtures, with `m40.eval.v1` unchanged:

- How many cases run?
- Is `task_success_rate` 1.0, and is `n_critical_fail` 0?
- Does `decide_release_gate` with `CANONICAL_POLICY` pass, and is
  `fail_reasons` empty?

Do not optimize either system first. Store the baseline.


In [ ]:
baseline = run_suite(pack)
baseline_gate = decide_release_gate(baseline, CANONICAL_POLICY)
expected = load_expected_payload()
print("case ids:", list(baseline.case_ids))
print("n:", baseline.n)
print("n_task_success:", baseline.n_task_success)
print("task_success_rate:", baseline.task_success_rate)
print("n_invariant_pass:", baseline.n_invariant_pass)
print("n_critical_fail:", baseline.n_critical_fail)
print("slice_fail_rates:", dict(baseline.slice_fail_rates))
print("family_success:", dict(baseline.family_success))
print("gate passed:", baseline_gate.passed)
print("fail_reasons:", list(baseline_gate.fail_reasons))
print("eval_version:", baseline.eval_version)
print("frozen n:", expected["n"], "frozen gate:", expected["baseline_gate_passed"])


A passing baseline is a freeze, not a proof that averages are enough.
The next experiments change one variable at a time against this pack.


## Deterministic graders

**Predict before running.** Timestamp your prediction.

One named change: invoke `rag-grounded-reset` with the M34 unsupported
citation defect. Schema, idempotency, and degraded-termination cases
stay healthy.

- Does `tool-schema-invalid` pass the schema grader while
  `execution_reached` is false, and what issue kind/field is in evidence?
- Does healthy idempotency report `effect_count == 1`?
- Does degraded fallback terminate as `degraded` with `degraded=True`?
- Does the injected citation fail, and what is `localized_failure`?
- Is coarse `outcome_success` still true for that answered-but-unsupported row?


In [ ]:
schema_case = pack.get("tool-schema-invalid")
schema_trace = invoke_case(schema_case)
schema_grade = grade_tool_schema(schema_case, schema_trace)
print("schema passed:", schema_grade.passed)
print("schema localized:", schema_grade.localized_failure)
print("schema issues:", schema_grade.evidence["issues"])
print("schema execution_reached:", schema_trace.execution_reached)

id_case = pack.get("tool-idempotency-replay")
id_trace = invoke_case(id_case)
id_grade = grade_idempotency(id_case, id_trace)
print("idempotency passed:", id_grade.passed)
print("idempotency effect_count:", id_trace.effect_count)
print("idempotency replayed:", id_trace.replayed)

deg_case = pack.get("agent-degraded-fallback")
deg_trace = invoke_case(deg_case)
deg_grade = grade_state_termination(deg_case, deg_trace)
print("termination passed:", deg_grade.passed)
print("termination terminal:", deg_trace.terminal)
print("termination degraded:", deg_trace.degraded)

cite_case = pack.get("rag-grounded-reset")
cite_broken = invoke_case(cite_case, defect="unsupported_citation")
cite_grade = grade_citation_support(cite_case, cite_broken)
print("citation passed:", cite_grade.passed)
print("citation localized:", cite_grade.localized_failure)
print("citation status:", cite_broken.status)
print("citation outcome_success:", outcome_success(cite_case, cite_broken))
print("citation support_ok:", cite_broken.support_ok)
print("verify_support ok:", verify_support(cite_broken.rag.answer, cite_broken.rag.pack).ok)


Failures should name an object: extra field, chunk id, terminal mismatch,
or `double_post:effect_count=2`. A blob score is not a grader.


## Rubric calibration

**Predict before running.** Timestamp your prediction.

Score the frozen hand-labeled set. Cases are frozen. The LLM judge
stays fail-closed.

- How many labels? How many disagreements?
- Which dimension disagrees (fluency versus an invariant)?
- Is `deterministic_required_for_invariants` true, and is the judge required?


In [ ]:
traces_by_source = {row.case_id: row.trace for row in baseline.rows}
rubric = calibrate_rubric(load_rubric_labels(), traces_by_source=traces_by_source)
print("n:", rubric["n"])
print("n_disagree:", rubric["n_disagree"])
print("disagreement_rate:", rubric["disagreement_rate"])
print("deterministic_required_for_invariants:", rubric["deterministic_required_for_invariants"])
print("llm_judge_required:", rubric["llm_judge_required"])
print("rows:")
for row in rubric["rows"]:
    print(row)
print("limit:", rubric["limit"])


Disagreement is a calibration signal. It is not permission to replace
citation, schema, termination, or idempotency with a judge.


## Critical slices versus aggregate success

**Predict before running.** Timestamp your prediction.

One named change: `inject_regression(..., defect="unsupported_citation")`
on the same pack.

- Is `task_success_rate` still 1.0?
- Is `critical_fail_rate` > 0?
- Is the citation slice fail rate 0.25 (one of four RAG citation cases)?


In [ ]:
injected = inject_regression(pack, defect="unsupported_citation")
print("task_success_rate:", injected.task_success_rate)
print("n_task_success:", injected.n_task_success)
print("critical_fail_rate:", injected.critical_fail_rate)
print("n_critical_fail:", injected.n_critical_fail)
print("slice_fail_rates:", dict(injected.slice_fail_rates))
print("slice_rates helper:", slice_rates(injected))
print("eval_version:", injected.eval_version)
print("case_ids unchanged:", list(injected.case_ids) == list(pack.case_ids))


Outcome success can stay high while a critical citation slice is not
zero. That is why release rules need slices, not only averages.


## Latency / steps / cost proxies

**Predict before running.** Timestamp your prediction.

On the **baseline** traces (not the injected suite):

- Is `mean_step_count` a positive deterministic value (not wall-clock)?
- Do RAG rows use three pipeline steps (retrieve, pack, synthesize)?
- Is `cost_proxy` derived from steps, effects, and packed chars?


In [ ]:
print("mean_step_count:", baseline.mean_step_count)
print("mean_cost_proxy:", baseline.mean_cost_proxy)
for row in baseline.rows:
    print(
        row.case_id,
        "family=",
        row.family,
        "steps=",
        row.trace.step_count,
        "cost=",
        row.trace.cost_proxy,
        "effects=",
        row.trace.effect_count,
        "packed_chars=",
        row.trace.packed_chars,
    )


Wall-clock milliseconds are observational noise in this fixture.
Steps, effects, and packed chars are the freezeable proxies.


In [ ]:
labels = ["baseline", "injected citation"]
task_rates = [baseline.task_success_rate, injected.task_success_rate]
crit_rates = [baseline.critical_fail_rate, injected.critical_fail_rate]
x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, task_rates, width, label="task success")
ax.bar(x + width / 2, crit_rates, width, label="critical fail")
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("rate")
ax.set_title("Does aggregate success hide a critical slice?")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.4)
fig.tight_layout()
plt.show()
print("baseline task/crit:", task_rates[0], crit_rates[0])
print("injected task/crit:", task_rates[1], crit_rates[1])


## Regression injection — same pack, canonical gate

**Predict before running.** Timestamp your prediction.

The injected suite already exists. One named change: apply
`CANONICAL_POLICY` (not the aggregate-only policy).

- Does the gate fail?
- What are the `fail_reasons` strings?
- Are `eval_version` and `case_ids` identical to the frozen pack?


In [ ]:
reg_gate = decide_release_gate(injected, CANONICAL_POLICY)
agg_gate = decide_release_gate(injected, AGGREGATE_ONLY_POLICY)
print("eval_version:", injected.eval_version)
print("pack eval_version:", pack.eval_version)
print("case_ids:", list(injected.case_ids))
print("pack case_ids:", list(pack.case_ids))
print("canonical passed:", reg_gate.passed)
print("canonical fail_reasons:", list(reg_gate.fail_reasons))
print("aggregate_only passed:", agg_gate.passed)
print("aggregate_only fail_reasons:", list(agg_gate.fail_reasons))


A regression suite that cannot fail is not a gate. The pack did not
change. The citation invariant did.


## Trace ablation

**Predict before running.** Timestamp your prediction.

Remove `used_memory_ids` from a captured observability dict.

- Which diagnosis becomes impossible?
- Does `citation_ids` remain, so unsupported-citation localization is
  still possible?


In [ ]:
obs = next(
    row.trace.observability()
    for row in injected.rows
    if row.case_id == "rag-grounded-reset"
)
memory_ablation = ablate_trace(obs, "used_memory_ids")
citation_ablation = ablate_trace(obs, "citation_ids")
print("removed memory field:", memory_ablation.removed)
print("blocked without used_memory_ids:", list(memory_ablation.blocked_diagnoses))
print("still possible:", list(memory_ablation.still_possible)[:4])
print("removed citation field:", citation_ablation.removed)
print("blocked without citation_ids:", list(citation_ablation.blocked_diagnoses))
print("citation_ids in remaining after memory ablation:", "citation_ids" in memory_ablation.remaining)


If a diagnosis needs a field, dropping that field is a product bug,
not a mystery. Observability is part of the eval contract.


## Code reading

Read `load_eval_pack`, `invoke_case`, `grade_tool_schema`,
`grade_citation_support`, `grade_state_termination`,
`grade_idempotency`, `decide_release_gate`, and `ablate_trace`.

**Predict before running.** Timestamp your prediction.

Predict:

- `load_eval_pack` with `require_canonical=True` on the tuned-dev pack
- `decide_release_gate` fail-reason strings on the injected suite
- which diagnosis `ablate_trace(..., "used_memory_ids")` blocks
- whether `optional_llm_judge` can be the sole required grader

Probe live objects: case ids, `fail_reasons`, `slice_fail_rates`.
Do not test substring membership of the word fail.


In [ ]:
print("=== load_eval_pack ===")
print(inspect.getsource(load_eval_pack))
print("=== invoke_case ===")
print(inspect.getsource(invoke_case))
print("=== grade_tool_schema ===")
print(inspect.getsource(grade_tool_schema))
print("=== grade_citation_support ===")
print(inspect.getsource(grade_citation_support))
print("=== grade_state_termination ===")
print(inspect.getsource(grade_state_termination))
print("=== grade_idempotency ===")
print(inspect.getsource(grade_idempotency))
print("=== decide_release_gate ===")
print(inspect.getsource(decide_release_gate))
print("=== ablate_trace ===")
print(inspect.getsource(ablate_trace))

print("canonical case ids:", list(pack.case_ids))
print("baseline fail_reasons:", list(baseline_gate.fail_reasons))
print("injected slice_fail_rates:", dict(injected.slice_fail_rates))
print("regression fail_reasons:", list(reg_gate.fail_reasons))
print("regression gate passed:", reg_gate.passed)
try:
    load_eval_pack(ROOT / "datasets" / "M40" / "contaminated_pack.json", require_canonical=True)
except EvalVersionError as exc:
    print("contaminated canonical load:", type(exc).__name__)
    print("contaminated canonical message:", str(exc))
print("memory ablation blocked:", list(memory_ablation.blocked_diagnoses))
hidden_row = next(row for row in injected.rows if row.case_id == "rag-grounded-reset")
print("localized citation:", hidden_row.grades[0].localized_failure)
print("m34 module:", type(hidden_row.trace.rag).__module__)
print("handoff v11_closed:", handoff_contract()["v11_closed"])


## Controlled failure — hidden critical

**Predict before running.** Timestamp your prediction.

A governance defect scores the injected-citation suite with an
aggregate-only gate.

- Does that gate pass?
- Is `n_critical_fail` 1 while `task_success_rate` stays 1.0?
- Are case ids still the canonical twelve?


In [ ]:
broken_hidden = pipeline_with_defect(defect="hidden_critical")
print("defect:", broken_hidden.defect)
print("claim:", broken_hidden.claim)
print("policy:", broken_hidden.policy_name)
print("aggregate passed:", broken_hidden.decision.passed)
print("fail_reasons:", list(broken_hidden.decision.fail_reasons))
print("task_success_rate:", broken_hidden.report.task_success_rate)
print("critical_fail_rate:", broken_hidden.report.critical_fail_rate)
print("n_critical_fail:", broken_hidden.report.n_critical_fail)
print("n:", broken_hidden.report.n)
print("case ids:", list(broken_hidden.report.case_ids))
print("eval_version:", broken_hidden.pack_version)
print("audit:", json.dumps(broken_hidden.audit, indent=2, default=str))


## Repair hidden-critical governance

**Predict before running.** Timestamp your prediction.

Call `repair_run` on `broken_hidden` only. Do not start a second
unrelated happy-path suite.

- Does the repaired canonical gate fail?
- Is the report the same object (same case ids, same critical rate)?
- Does `broken_hidden.decision.passed` stay true?


In [ ]:
repaired_hidden = repair_run(broken_hidden)
print("repaired defect:", repaired_hidden.defect)
print("repaired claim:", repaired_hidden.claim)
print("repaired policy:", repaired_hidden.policy_name)
print("repaired passed:", repaired_hidden.decision.passed)
print("repaired fail_reasons:", list(repaired_hidden.decision.fail_reasons))
print("same report object:", repaired_hidden.report is broken_hidden.report)
print("broken still aggregate-passed:", broken_hidden.decision.passed)
print("broken critical_fail_rate:", broken_hidden.report.critical_fail_rate)


The repair is the gate, not the model. The citation defect is still
on the broken report. Canonical slices refuse to ship it.


## Controlled failure — contaminated pack

**Predict before running.** Timestamp your prediction.

A second named governance defect loads a pack that was tuned against.

- What is `eval_version`?
- Is `contaminated` true, and is `n` less than 12?
- Are `rag-holdout-email` and `tool-idempotency-replay` absent?
- Does the aggregate-only gate still pass?


In [ ]:
broken_contam = pipeline_with_defect(defect="contaminated_pack")
print("defect:", broken_contam.defect)
print("claim:", broken_contam.claim)
print("eval_version:", broken_contam.pack_version)
print("contaminated:", broken_contam.report.pack_contaminated)
print("tuned_against:", broken_contam.report.tuned_against)
print("n:", broken_contam.report.n)
print("case ids:", list(broken_contam.report.case_ids))
print("aggregate passed:", broken_contam.decision.passed)
print("holdout email present:", "rag-holdout-email" in broken_contam.report.case_ids)
print("idempotency present:", "tool-idempotency-replay" in broken_contam.report.case_ids)
print("audit:", json.dumps(broken_contam.audit, indent=2, default=str))


## Repair contamination from the broken object

**Predict before running.** Timestamp your prediction.

Call `repair_run` on `broken_contam` only.

- Does the repaired pack version become `m40.eval.v1` with `n == 12`?
- Is `repaired.report.pack_contaminated` false?
- Does `broken_contam.pack_version` stay `m40.eval.tuned-dev`?


In [ ]:
repaired_contam = repair_run(broken_contam)
print("repaired defect:", repaired_contam.defect)
print("repaired claim:", repaired_contam.claim)
print("repaired version:", repaired_contam.pack_version)
print("repaired n:", repaired_contam.report.n)
print("repaired contaminated:", repaired_contam.report.pack_contaminated)
print("repaired passed:", repaired_contam.decision.passed)
print("repaired case ids:", list(repaired_contam.report.case_ids))
print("broken version still:", broken_contam.pack_version)
print("broken n still:", broken_contam.report.n)


Reloading the clean versioned set is the repair. Editing gold labels
until the average looks good is the defect.


## Evidence contract

Submit timestamped predictions, the baseline suite, localized grader
strings, rubric disagreement, slice-versus-average numbers, the
regression gate, an ablation diagnosis, and both governance repairs
from broken objects.

Do not prefill [UNFILLED BY LEARNER] fields. A green notebook is not
competence.


## No-AI gate

Complete `missions/M40/no_ai_gate.md` from a blank page. Fresh
numbers: Atlas of Rivers / QH 441 / floor 4 / BIN-12 occupancy 3 /
refund 8802 / a sonnet about forklifts.

Five eval cases, deterministic versus rubric, a slice/aggregate
report, one contamination example, one average-independent blocker.
Leave the repository copy unfilled.


## ADR and formal review

Fill `missions/M40/adr_prompt.md` using `templates/ADR.md`. The
decision is V11 evaluation governance: dataset ownership/versioning,
grader hierarchy, severities, regression baseline, release thresholds,
human-review triggers, and audit retention.

Status, date, owner, and decision stay **[UNFILLED BY LEARNER]** here.

Formal engineering review uses `missions/M40/review_brief.md`. This
branch does not mark M40 repository-executable.


## Handoff to M41 / M42

M41 receives a versioned suite, a failure taxonomy, gates, and
observability expectations. M42 may reuse the suite.

M41 is integrated architecture. This notebook does not teach that
diagram. **V11 does not close** because this package exists. P7
phase-end is the evaluation opening, not learner completion.


## Mission summary

You froze cases before optimizing, localized deterministic failures,
calibrated a rubric without replacing invariants, watched an average
hide a critical slice, failed a regression gate on an unchanged pack,
and repaired eval governance from broken objects.

This fixture is **not a production** eval platform. Do not import a
paid eval SDK. Do not retune M34 or M39 to fit the scores.


In [ ]:
assert baseline.eval_version == EVAL_VERSION == "m40.eval.v1"
assert baseline.n == 12 and baseline.n_task_success == 12
assert baseline.n_critical_fail == 0
assert baseline_gate.passed is True
assert list(baseline_gate.fail_reasons) == []
assert schema_grade.passed is True and schema_trace.execution_reached is False
assert "extra:currency" in schema_grade.evidence["issues"]
assert id_grade.passed is True and id_trace.effect_count == 1
assert deg_trace.terminal == "degraded" and deg_trace.degraded is True
assert cite_grade.passed is False
assert cite_grade.localized_failure == "unsupported_claim:doc-account-access::c0"
assert outcome_success(cite_case, cite_broken) is True
assert rubric["n_disagree"] == 1 and rubric["deterministic_required_for_invariants"] is True
assert injected.n_task_success == 12 and injected.n_critical_fail == 1
assert injected.eval_version == pack.eval_version
assert list(injected.case_ids) == list(pack.case_ids)
assert reg_gate.passed is False
assert "slice:citation_support:0.250>0.0" in reg_gate.fail_reasons
assert agg_gate.passed is True
assert memory_ablation.blocked_diagnoses == ("stale_memory_posted_as_complete",)
assert broken_hidden.decision.passed is True
assert repaired_hidden.decision.passed is False
assert repaired_hidden.report is broken_hidden.report
assert broken_contam.pack_version == "m40.eval.tuned-dev"
assert repaired_contam.pack_version == "m40.eval.v1"
assert repaired_contam.report.n == 12
assert broken_contam.report.n < 12
assert type(hidden_row.trace.rag).__module__ == "missions.M34.rag_pipeline"
assert handoff_contract()["v11_closed"] is False
print("M40 integrity checks passed")
print(handoff_contract()["handoff"])
